# v2 RL — analysis

Loads the PPO agent trained by `train.py` and, without ever retraining:

1. plots the learning curve,
2. runs the **same-game gate** — your v2 DP policy driven through this env must reproduce your v2 numbers,
3. scores PPO against the v2 DP optimum and the always-BdC baseline,
4. traces a few games to see what PPO actually does.

**Needs, in this folder:** `fx_mechanics.py`, `v2rl_env.py`, your full `v2_sequential_offers.py`, and a trained run under `runs/<RUN_NAME>/` (run `train.py` first).


In [ ]:
import os, pickle
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

from v2rl_env import V2Game, GameSpec, Z_OFFER
import v2_sequential_offers as v2
import rl_diagnostics as diag

RUN_NAME = "ppo_v2rl"                 # matches train.py --run-name
RUN_DIR  = f"results/{RUN_NAME}"

spec_env = GameSpec()                 # the T1 card, RL side
L, sd, a0 = spec_env.L, spec_env.params.sd, spec_env.params.a0
V2_SPEC  = v2.TraderSpec()            # the same card, DP side

model = PPO.load(f"{RUN_DIR}/ppo_final")     # best_model.zip is also available
print("loaded", f"{RUN_DIR}/ppo_final.zip")

## 1. Learning curve

Written by `train.py`'s eval callback. Reward is scaled ×100 during training, so these values are already in **% P/L**.

In [ ]:
ev = np.load(f"{RUN_DIR}/evaluations.npz")
ts, pl = ev["timesteps"], ev["results"].mean(axis=1)

plt.figure(figsize=(8, 4.5))
plt.plot(ts, pl, marker="o", lw=1.5, label="PPO (eval mean)")
plt.axhline(-1.46, ls=":",  color="grey",      label="always-BdC  (-1.46%)")
plt.axhline(-0.36, ls="--", color="tab:green", label="v2 DP optimum (-0.36%)")
plt.xlabel("timesteps"); plt.ylabel("P/L  (%)")
plt.title("PPO learning curve"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 2. Same-game gate

Solve your v2 DP, then drive **this** env with the DP's policy on the DP's **own** rate paths. If the env plays the same game, the mean P/L is identical and each path's wealth matches to floating-point level. This also pins the −0.36% target the RL agent is chasing.

The belief impedance mismatch (v2's beliefs are grid indices, the env's are continuous) is bridged by tracking grid indices in parallel, synced from whichever side of the env's bracket moved after each offer.

*Full-resolution solve — a few minutes the first time, then cached to `runs/v2_sol.pkl`.*


In [ ]:
def eval_dp_on_paths(sol, Xs, a5s):
    """Play v2's grid DP policy through the continuous env on forced paths.
    Returns per-path terminal wealth (GBP)."""
    g, env, K = sol.grids, V2Game(spec_env), spec_env.K
    W = np.empty(len(Xs))
    for i in range(len(Xs)):
        env.reset(seed=0, options={"path": Xs[i], "a5": float(a5s[i])})
        lo, hi, done, r = v2.NO_FLOOR, g.n_offer, False, 0.0
        while not done:
            if env.phase == 0:                       # offer step
                if env.k == K:                       # round start -> fresh belief
                    lo, hi = v2.NO_FLOOR, g.n_offer
                act, _ = v2.greedy_offer_action(sol, env.n, env.k, lo, hi,
                                                 env.c, env.d, env.a)
                if act is None:                      # v2 stops -> no-op probe
                    _, r, te, tr, _ = env.step(np.array([0.0, 0.0], np.float32))
                else:
                    pi, q = act
                    z = g.z[pi]                       # v2 grid price in sds from anchor
                    a_size  = (q / env.c) if env.c > 1e-12 else 0.0
                    lo0, hi0 = env.lo, env.hi
                    _, r, te, tr, _ = env.step(np.array([z, a_size], np.float32))
                    if   env.lo > lo0: lo = pi        # floor rose  -> accepted
                    elif env.hi < hi0: hi = pi        # ceiling fell -> rejected
            else:                                    # BdC step: X revealed
                m = v2.greedy_bdc_action(sol, env.n, env.c, env.d, env.X)
                a_price = (m / env.c) if env.c > 1e-12 else 0.0
                _, r, te, tr, _ = env.step(np.array([0.0, a_price], np.float32))
            done = te or tr
        W[i] = r * L + L
    return W

cache = f"{RUN_DIR}/v2_sol.pkl"
if os.path.exists(cache):
    sol = pickle.load(open(cache, "rb"))
    print("loaded cached v2 solution")
else:
    sol = v2.solve_v2(V2_SPEC, print_progress=True)   # default grids = full res
    pickle.dump(sol, open(cache, "wb"))
print(f"v2 DP value: {sol.pl()*100:+.3f}% P/L")

N = 8000
sim   = v2.simulate_paths(sol, n_paths=N, seed=1)          # v2's play + paths
W_env = eval_dp_on_paths(sol, sim["X"], sim["a5"])         # same paths, our env
pl_v2, pl_env = (sim["W"] - L) / L, (W_env - L) / L
print(f"v2 simulate_paths : {pl_v2.mean()*100:+.3f}%")
print(f"env DP-replay     : {pl_env.mean()*100:+.3f}%")
print(f"max per-path |dW| : {np.abs(W_env - sim['W']).max():.2e} GBP   (wealth ~ {L:,.0f})")

## 3. Scorecard

Only the learned policy is measured by simulation — it has no closed form. The DP optimum and always-BdC are exact values, so they are quoted rather than re-simulated.

In [ ]:
rng   = np.random.default_rng(2024)
rates = a0 + sd * np.cumsum(rng.standard_normal((N, spec_env.rounds + 1)), axis=1)
Xs, a5s = rates[:, :spec_env.rounds], rates[:, spec_env.rounds]

def eval_ppo_on_paths(model, Xs, a5s):
    env, pl = V2Game(spec_env), np.empty(len(Xs))
    for i in range(len(Xs)):
        obs, _ = env.reset(seed=0, options={"path": Xs[i], "a5": float(a5s[i])})
        done, r = False, 0.0
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, r, te, tr, _ = env.step(a); done = te or tr
        pl[i] = r
    return pl

# Only the learned policy is measured by simulation -- it has no closed form.
# The DP optimum is exact (sol.pl()) and always-BdC is a fixed constant, so both are
# quoted, not re-simulated.
pl_ppo = eval_ppo_on_paths(model, Xs, a5s)

def row(name, x):
    print(f"{name:<26}{x.mean()*100:+8.3f}%  +/- {2*x.std()/np.sqrt(len(x))*100:.3f}%")
def const_row(name, pct):
    print(f"{name:<26}{pct:+8.3f}%        exact")

print(f"{'strategy':<26}{'P/L':>9}   2 s.e.")
print("-" * 50)
row("PPO (trained)", pl_ppo)
const_row("v2 DP optimum", sol.pl() * 100)
const_row("always-BdC", diag.bdc_floor_pct(spec_env))

### DQN on v2's grid (optional)

If you've run `train_dqn.py`, this scores that agent on the **same** paths as the table above, so DQN, PPO, the DP and the baselines are all directly comparable. DQN plays on v2's exact action grid, so it's the closest model-free counterpart to the DP.

In [ ]:
import os
from v2rl_env import GridV2Game

dqn_dir = "results/dqn_v2rl"
if os.path.exists(os.path.join(dqn_dir, "dqn_final.zip")):
    from stable_baselines3 import DQN
    dqn = DQN.load(os.path.join(dqn_dir, "dqn_final"))
    genv = GridV2Game(spec_env)
    pl_dqn = np.empty(len(Xs))
    for i in range(len(Xs)):
        obs, _ = genv.reset(seed=0, options={"path": Xs[i], "a5": float(a5s[i])})
        done, r = False, 0.0
        while not done:
            a, _ = dqn.predict(obs, deterministic=True)
            obs, r, te, tr, _ = genv.step(a); done = te or tr
        pl_dqn[i] = r
    print(f"{'strategy':<26}{'P/L':>9}   2 s.e.")
    print("-" * 50)
    row("PPO (continuous)", pl_ppo)
    row("DQN (v2 grid)", pl_dqn)
    const_row("v2 DP optimum", sol.pl() * 100)
    const_row("always-BdC", diag.bdc_floor_pct(spec_env))
else:
    print("No DQN run at results/dqn_v2rl/ — run train_dqn.py to add it here.")

## 4. What did PPO learn?

Deterministic play, offer by offer. `X` is the hidden true rate (peeked here only to show whether each offer would clear). Watch whether it genuinely trades with the MM or just carries pounds to the BdC dump.

In [ ]:
def trace_ppo(model, seed=None):
    env = V2Game(spec_env)
    obs, _ = env.reset(seed=seed)
    rounds, cur, done, r = [], {"n": env.n, "a": env.a, "X": env.X, "offers": []}, False, 0.0
    while not done:
        ph = env.phase
        a, _ = model.predict(obs, deterministic=True)
        if ph == 0:
            z = float(np.clip(a[0], -Z_OFFER, Z_OFFER))
            P = env.a + env.sd * z
            q = np.clip(float(a[1]), 0, 1) * env.c
            acc = P < env.X
            obs, r, te, tr, _ = env.step(a)
            cur["offers"].append((P, z, q, acc, env.c, env.d))
        else:
            m = np.clip(float(a[1]), 0, 1) * env.c
            obs, r, te, tr, _ = env.step(a)
            cur.update(bdc=m, c_end=env.c, d_end=env.d); rounds.append(cur)
            if not (te or tr):
                cur = {"n": env.n, "a": env.a, "X": env.X, "offers": []}
        done = te or tr
    return rounds, r

def show_trace(rounds, pl):
    for rd in rounds:
        print(f"Round {rd['n']}:  anchor {rd['a']:.4f}   hidden X {rd['X']:.4f}")
        if not rd["offers"]:
            print("   (no offer made)")
        for (P, z, q, acc, c, d) in rd["offers"]:
            tag = "ACCEPT" if acc else "reject"
            print(f"   offer P={P:.4f} (z{z:+.2f})  sell GBP {q:>8,.0f}  {tag}"
                  f"   -> c={c:>8,.0f}  d=${d:>11,.0f}")
        print(f"   BdC dump GBP {rd.get('bdc', 0):>8,.0f}"
              f"   -> c={rd['c_end']:>8,.0f}  d=${rd['d_end']:>11,.0f}")
    print(f"==>  P/L {pl*100:+.2f}%\n")

for s in (1, 7, 13):
    print(f"----- game (seed {s}) -----")
    show_trace(*trace_ppo(model, seed=s))